In [ ]:
!pip install ipywidgets

In [1]:
import ipywidgets as widgets
from IPython.display import display

# Screen size used by Verilog
X_SIZE = 640
Y_SIZE = 480

# Default register values from Verilog
DEFAULT_ZR0 = -8192
DEFAULT_ZI0 = -6144
DEFAULT_STEP = 26
DEFAULT_MAXIT = 30


# Fake write_view for testing without the PYNQ board
# This does NOT write to FPGA registers.
# It only prints the values that would be written.
def write_view(zr0, zi0, step, maxit):
    print("Fake FPGA register write:")
    print("ZR0   =", zr0)
    print("ZI0   =", zi0)
    print("STEP  =", step)
    print("MAXIT =", maxit)


# Set the fractal view using intuitive zoom/pan values
# zoom = 1.0 means default zoom
# zoom = 2.0 means 2x zoom in
# pan_x and pan_y are in pixels
def set_view(zoom=1.0, pan_x=0, pan_y=0, maxit=30):

    # Convert zoom into STEP
    # Larger zoom means smaller step
    step = int(round(DEFAULT_STEP / zoom))

    # Prevent step becoming zero
    if step < 1:
        step = 1

    # Find default centre of the image in Q12 units
    centre_r = DEFAULT_ZR0 + (X_SIZE // 2) * DEFAULT_STEP
    centre_i = DEFAULT_ZI0 + (Y_SIZE // 2) * DEFAULT_STEP

    # Apply pan
    # pan_x = 1 means shift centre by 1 pixel horizontally
    # pan_y = 1 means shift centre by 1 pixel vertically
    centre_r = centre_r + pan_x * step
    centre_i = centre_i + pan_y * step

    # Calculate new top-left coordinate
    zr0 = centre_r - (X_SIZE // 2) * step
    zi0 = centre_i - (Y_SIZE // 2) * step

    # Make sure all register values are integers
    zr0 = int(round(zr0))
    zi0 = int(round(zi0))
    step = int(step)
    maxit = int(maxit)

    # Fake register write
    write_view(zr0, zi0, step, maxit)


# Create sliders
zoom_slider = widgets.FloatSlider(
    value=1.0,
    min=0.5,
    max=20.0,
    step=0.1,
    description="Zoom",
    continuous_update=False
)

pan_x_slider = widgets.IntSlider(
    value=0,
    min=-320,
    max=320,
    step=1,
    description="Pan X",
    continuous_update=False
)

pan_y_slider = widgets.IntSlider(
    value=0,
    min=-240,
    max=240,
    step=1,
    description="Pan Y",
    continuous_update=False
)

maxit_slider = widgets.IntSlider(
    value=30,
    min=1,
    max=63,
    step=1,
    description="Max Iter",
    continuous_update=False
)

apply_button = widgets.Button(description="Apply View")
reset_button = widgets.Button(description="Reset View")
output = widgets.Output()

def apply_view(button):
    with output:
        output.clear_output()

        set_view(
            zoom=zoom_slider.value,
            pan_x=pan_x_slider.value,
            pan_y=pan_y_slider.value,
            maxit=maxit_slider.value
        )

def reset_view(button):
    zoom_slider.value = 1.0
    pan_x_slider.value = 0
    pan_y_slider.value = 0
    maxit_slider.value = DEFAULT_MAXIT

    with output:
        output.clear_output()

        set_view(
            zoom=1.0,
            pan_x=0,
            pan_y=0,
            maxit=DEFAULT_MAXIT
        )

apply_button.on_click(apply_view)
reset_button.on_click(reset_view)

display(
    zoom_slider,
    pan_x_slider,
    pan_y_slider,
    maxit_slider,
    widgets.HBox([apply_button, reset_button]),
    output
)

FloatSlider(value=1.0, continuous_update=False, description='Zoom', max=20.0, min=0.5)

IntSlider(value=0, continuous_update=False, description='Pan X', max=320, min=-320)

IntSlider(value=0, continuous_update=False, description='Pan Y', max=240, min=-240)

IntSlider(value=30, continuous_update=False, description='Max Iter', max=63, min=1)

Output()